# Dataset download 

In [ ]:
import torch
from torchvision import datasets, transforms
from PIL import Image
import os

def generate_mnist_samples(number: int, max_samples: int = 100, output_dir: str = "../../tests/generated_samples") -> None:
    """
    Generate and save MNIST samples for a specified number.

    Args:
        number (int): The MNIST digit to generate samples for (0-9).
        max_samples (int, optional): The maximum number of samples to generate. Defaults to 100.
        output_dir (str, optional): The base output directory. Defaults to "../../tests/generated_samples".

    Returns:
        None
    """
    # Set up the output directory
    digit_output_dir = os.path.join(output_dir, f"mnist_{number}")
    os.makedirs(digit_output_dir, exist_ok=True)

    # Download and load MNIST dataset
    transform = transforms.Compose([transforms.ToTensor()])
    mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

    # Filter for only the specified number
    filtered_dataset = [(img, label) for img, label in mnist_train if label == number]

    # Generate and save images
    for i, (img, _) in enumerate(filtered_dataset[:max_samples]):
        # Convert tensor to PIL Image
        pil_img = transforms.ToPILImage()(img.squeeze())
        
        # Resize to 100x100
        pil_img = pil_img.resize((100, 100), Image.BILINEAR)
        
        # Save the image
        pil_img.save(os.path.join(digit_output_dir, f"mnist_{number}_{i:05d}.png"))

    print(f"Generated {min(len(filtered_dataset), max_samples)} images of the number {number} in {digit_output_dir}")

In [ ]:
# for i in range(10):
#     generate_mnist_samples(i, max_samples=10)

In [ ]:
import os
import random
from PIL import Image
from torchvision import transforms

def generate_rotated_samples(input_dir: str, angle_range: tuple[int, int] = (-30, 30), shift_percent: float = 0.1, images_count: int = 3) -> None:
    """
    Generate additional rotated and shifted versions of each image in the input directory.
    Images will have a black background.
    
    Args:
        input_dir (str): Directory containing the original images
        angle_range (tuple[int, int]): Range of angles for random rotation (min, max)
        shift_percent (float): Maximum shift as a percentage of image dimensions
        images_count (int): Number of augmented images to generate per original image
        
    Returns:
        None
    """
    if not os.path.exists(input_dir):
        raise ValueError(f"Input directory {input_dir} does not exist")
    
    for filename in os.listdir(input_dir):
        if not filename.endswith(('.png', '.jpg', '.jpeg')):
            continue
            
        image_path = os.path.join(input_dir, filename)
        original_image = Image.open(image_path)
        width, height = original_image.size
        
        # Calculate maximum pixel shift based on percentage
        max_shift_x = int(width * shift_percent)
        max_shift_y = int(height * shift_percent)
        
        # Generate augmented versions
        for i in range(images_count):
            # Create a new image with black background (slightly larger to accommodate rotation and shift)
            padding = max(max_shift_x, max_shift_y) + int(max(width, height) * 0.2)
            new_width, new_height = width + 2*padding, height + 2*padding
            new_image = Image.new('L', (new_width, new_height), 0)
            
            # Paste original image in center
            new_image.paste(original_image, (padding, padding))
            
            # Apply random rotation
            angle = random.uniform(angle_range[0], angle_range[1])
            rotated_image = new_image.rotate(angle, resample=Image.BILINEAR, fillcolor=0)
            
            # Apply random shift
            shift_x = random.randint(-max_shift_x, max_shift_x)
            shift_y = random.randint(-max_shift_y, max_shift_y)
            
            # Crop to original size but with shift
            crop_left = padding + shift_x
            crop_top = padding + shift_y
            crop_right = crop_left + width
            crop_bottom = crop_top + height
            
            final_image = rotated_image.crop((crop_left, crop_top, crop_right, crop_bottom))
            
            # Create new filename with augmentation indicator
            name, ext = os.path.splitext(filename)
            new_filename = f"{name}_aug{i+1}{ext}"
            new_path = os.path.join(input_dir, new_filename)
            
            # Save augmented image
            final_image.save(new_path)
    
    print(f"Generated rotated and shifted samples in {input_dir}")

def remove_generated_rotations(input_dir: str) -> None:
    """
    Remove all generated augmented images from the specified directory.
    
    Args:
        input_dir (str): Directory containing the images
        
    Returns:
        None
    """
    if not os.path.exists(input_dir):
        raise ValueError(f"Input directory {input_dir} does not exist")
        
    removed_count = 0
    for filename in os.listdir(input_dir):
        if "_rot" in filename or "_aug" in filename:
            file_path = os.path.join(input_dir, filename)
            os.remove(file_path)
            removed_count += 1
    
    print(f"Removed {removed_count} augmented images from {input_dir}")

# Example usage:
base_dir = "../../tests/prepared_samples/"

In [ ]:
classes_to_subclasses = {
    # 0: [1],
    # 1: [1, 2, 3],
    # 2: [1, 2],
    3: [1],
    # 4: [1, 2],
    # 5: [1],
    # 6: [1],
    # 7: [1, 2],
    # 8: [1, 2],
    # 9: [1, 2],
}


In [ ]:
for class_num in classes_to_subclasses:
    for subclasses in classes_to_subclasses[class_num]:
        digit_dir = os.path.join(base_dir, str(class_num) + '_' + str(subclasses))
        remove_generated_rotations(digit_dir)
        generate_rotated_samples(digit_dir, angle_range=(-10, 10), shift_percent=0.1, images_count=5)